# Chapitre 2 : Reproductibilité et développement professionnel

*MLSMM2151 Data Science · Séance 1, vendredi 13 novembre, matin · Corentin Vande Kerckhove*

> **CONSEIL** : commencez par enregistrer une copie de ce notebook dans votre Google Drive
> (`Fichier` > `Enregistrer une copie dans Drive`) pour pouvoir prendre vos propres notes.

## Notebook 1 sur 2 : structurer un projet et ses données

**Durée estimée : 25 minutes.**

### Objectifs

À la fin de ce notebook, vous saurez :

- organiser un dossier de projet de data science ;
- dire pourquoi on ne met **jamais** de données dans un dépôt git ;
- charger des données dans Colab depuis trois sources : une adresse web, votre
  Google Drive, votre OneDrive UCLouvain ;
- faire en sorte que votre travail survive à la fermeture de Colab.

## 2.1. Organisation d'un projet de Data Science

Un projet mal rangé coûte du temps à tout le monde, y compris à vous dans trois
semaines. La structure ci-dessous est un standard raisonnable, et c'est celle
qu'on attend dans les devoirs.

```
mon-projet/
├── README.md            Ce que fait ce projet, et comment le faire tourner
├── requirements.txt     Les bibliothèques nécessaires, avec leurs versions
├── .gitignore           Ce que git doit ignorer, à commencer par les données
├── data/
│   └── README.md        D'où viennent les données, PAS les données elles-mêmes
├── notebooks/           L'exploration, dans l'ordre : 01-, 02-, 03-
└── src/                 Le code réutilisable, en fichiers .py
```

Trois principes derrière cette structure.

**Quelqu'un doit pouvoir refaire tourner votre travail sans vous.** C'est la
définition opérationnelle de la reproductibilité. Le test est simple : sur une
machine vierge, avec seulement votre dépôt, est-ce que ça tourne ?

**Le code se sépare de l'exploration.** Une fonction qu'on réutilise dans trois
notebooks n'a rien à faire copiée trois fois : elle va dans `src/`.

**On numérote ce qui a un ordre.** `01-exploration.ipynb`, `02-nettoyage.ipynb`.
Six mois plus tard, personne ne se souvient de l'ordre, y compris vous.

## 2.2. Gestion des dépendances

Votre code dépend de bibliothèques, et ces bibliothèques changent. Un notebook
qui tournait il y a un an peut échouer aujourd'hui parce que pandas a modifié un
comportement.

La réponse minimale est un fichier `requirements.txt` :

```
pandas>=2.0
scikit-learn>=1.4
matplotlib>=3.7
```

qu'on réinstalle avec `pip install -r requirements.txt`.

Dans Colab, la plupart de ces bibliothèques sont déjà là. Vérifiez toujours les
versions avec lesquelles vous travaillez : c'est la première chose à regarder
quand un code qui marchait ne marche plus.

In [ ]:
import sys
import pandas as pd
import sklearn
import matplotlib

print(f"Python       {sys.version.split()[0]}")
print(f"pandas       {pd.__version__}")
print(f"scikit-learn {sklearn.__version__}")
print(f"matplotlib   {matplotlib.__version__}")

### Hack-Time 🛠️

Vous venez d'afficher les versions de trois bibliothèques. Le dépôt de votre
devoir 1 contient ce `requirements.txt` :

```
pandas>=2.0
numpy>=1.24
matplotlib>=3.7
scikit-learn>=1.4
```

Un camarade y ajoute un notebook qui commence par `import xgboost` et
`import seaborn`, et vous assure que « ça tourne ». Dans la cellule
ci-dessous :

1. Vérifiez si ces deux bibliothèques sont présentes dans votre session, et
   en quelle version. Une piste : `import seaborn` puis
   `seaborn.__version__`.
2. Dites ce qu'il faudrait ajouter à `requirements.txt` pour que le
   correcteur, sur une machine vierge, puisse exécuter ce notebook.
3. En une phrase : pourquoi « ça tourne chez moi » n'est-il pas un argument ?

**Trois minutes.**

**Défi optionnel**
Affichez `!pip list | wc -l` : combien de paquets Colab a-t-il installés pour
vous ? Faut-il tous les mettre dans `requirements.txt` ? Pourquoi non ?

In [ ]:
# Votre code ici

## 2.3. Notebooks contre scripts Python

Ce ne sont pas des concurrents, ce sont deux outils pour deux moments.

| | Notebook `.ipynb` | Script `.py` |
|---|---|---|
| Fait pour | Explorer, montrer, expliquer | Exécuter, réutiliser, mettre en production |
| Force | On voit les résultats à côté du code | Testable, importable, lisible dans un `git diff` |
| Faiblesse | État caché, ordre d'exécution trompeur | Aucun résultat visible sans le lancer |

Le vrai piège du notebook est **l'état caché**. Vous définissez une variable,
vous supprimez la cellule qui la définissait, et tout continue de fonctionner.
Chez vous. Le lendemain, chez quelqu'un d'autre, plus rien ne marche.

**La règle à appliquer avant tout rendu** : `Exécution` > `Redémarrer et tout
exécuter`. Si ça casse, ce n'est pas prêt. C'est un critère explicite des
grilles de correction de ce cours.

En pratique : explorez dans un notebook, et dès qu'un bout de code est stabilisé
et réutilisé, déplacez-le dans un `.py`.

### Hack-Time 🛠️

Vous venez de lire la règle : on explore dans un notebook, on stabilise dans
un `.py`. Voici trois bouts de code. Pour chacun, dites s'il mérite un
fichier `.py`, et pourquoi.

1. Une fonction `nettoyer_prix(colonne)` qui transforme `$64.00` en `64.0`,
   copiée-collée dans trois notebooks.
2. La cellule `annonces.head()`.
3. Une cellule de trente lignes qui produit la figure 2 de votre rapport,
   avec ses couleurs et ses titres.

Puis une question sur votre propre dépôt de devoir : pourquoi `sujet.py` vous
est-il fourni en fichier `.py`, et non en cellule de votre notebook ?

**Trois minutes.** Notez vos réponses dans la cellule ci-dessous.

**Défi optionnel**
Vous mettez `nettoyer_prix` dans `src/nettoyage.py`. Comment votre notebook
l'utilise-t-il ensuite ? Et que se passe-t-il quand ce notebook est ouvert
depuis GitHub dans Colab, où rien n'est cloné ?

*Vos réponses :*

1.
2.
3.

`sujet.py` :

## 2.4. Où vivent les données

Nous arrivons au point le plus important de ce notebook, et à une habitude qu'il
faut prendre maintenant parce qu'elle est très difficile à corriger après coup.

> **On ne met jamais de données dans un dépôt git.**

### Pourquoi

1. **Le poids.** Git garde l'historique complet : un CSV de 50 Mo modifié dix
   fois, c'est un dépôt de 500 Mo, pour toujours.
2. **Git ne sait pas versionner des données.** Un `git diff` sur un CSV est
   illisible, sur un `.xlsx` il est vide.
3. **Les licences.** Beaucoup de jeux de données s'utilisent mais ne se
   redistribuent pas.
4. **Les données personnelles.** Un dépôt poussé en public avec des données
   clients, c'est un incident RGPD.

Les quatre raisons sont développées dans le
[`data/README.md`](https://github.com/cvandekerckh/mlsmm2151/blob/main/data/README.md)
du cours. Le point 4 est le plus sérieux. Vous allez, dans votre vie professionnelle,
manipuler des données que vous n'avez pas le droit de diffuser. L'habitude prise
sur un jeu de données Airbnb public est celle que vous appliquerez le jour où
ça comptera vraiment.

### Comment on l'applique mécaniquement

Un fichier `.gitignore` à la racine du projet, qui interdit à git de voir ces
fichiers même si vous les oubliez :

```
*.csv
*.parquet
*.xlsx
*.pkl
```

C'est ce que fait le dépôt de ce cours. Le dépôt contient le **code** qui va
chercher les données ; il ne contient jamais les **données**.

### Les trois endroits où vos données peuvent vivre

| Ce que c'est | Où ça va | Comment on y accède depuis Colab |
|---|---|---|
| Un jeu de données public, distribué avec un projet | Une **adresse web** stable | `pd.read_csv(url)` |
| Votre travail : fichiers intermédiaires, modèles, exports | Votre **Google Drive** | on monte le Drive |
| Des données d'entreprise ou confidentielles | Votre **OneDrive UCLouvain** | lien de partage en téléchargement direct |

Il y a une contrainte technique à garder en tête pendant tout le cours.

> **Un notebook ouvert depuis GitHub dans Colab est éphémère.** Rien n'est
> cloné, il n'y a pas de fichier local, et tout disparaît quand vous fermez
> l'onglet.

C'est exactement pour cela que les trois solutions ci-dessus sont des solutions :
aucune ne suppose qu'un fichier soit déjà présent sur la machine.

![Où vivent le code et les données](https://raw.githubusercontent.com/cvandekerckh/mlsmm2151/main/images/ch02-ou-vivent-les-donnees.png)

*Le code dans le dépôt, les données ailleurs, et une session Colab qui ne retient rien.*


### Cas 1 : depuis une adresse web

C'est ce que nous avons fait depuis le début de la séance. Les données du cours
sont publiées en *asset de release* du dépôt GitHub, ce qui les place à côté du
dépôt sans les mettre dedans.

In [ ]:
import pandas as pd

BASE = "https://github.com/cvandekerckh/mlsmm2151/releases/download/v1-donnees"

annonces = pd.read_csv(f"{BASE}/airbnb_bruxelles_raw.csv", low_memory=False)
print(f"{annonces.shape[0]} annonces chargées depuis le web")

Simple, mais avec un défaut : le fichier est retéléchargé à chaque session.
C'est acceptable pour 5 Mo, beaucoup moins pour 500 Mo.

### Cas 2 : votre Google Drive

Monter votre Drive donne à Colab l'accès à vos fichiers personnels. Ils
survivent à la fermeture de la session, et vous les retrouvez d'un notebook à
l'autre.

```python
from google.colab import drive
drive.mount('/content/drive')

# Ensuite, des chemins tout à fait normaux :
mon_fichier = "/content/drive/MyDrive/MLSMM2151/data/mon_travail.csv"
donnees = pd.read_csv(mon_fichier)
```

Colab vous demandera une autorisation. C'est normal : vous donnez à un notebook
l'accès à votre Drive, autant savoir ce que le notebook fait.

**C'est ici que va votre travail personnel** : tableaux intermédiaires, modèles
entraînés, exports. Pas dans le dépôt git.

Le cours fournit une fonction qui automatise tout cela.

In [ ]:
# Cette ligne récupère le fichier utils/donnees.py du dépôt du cours.
!curl -fsSL -o donnees.py https://raw.githubusercontent.com/cvandekerckh/mlsmm2151/main/utils/donnees.py

from donnees import charger

charger.disponibles()

`charger` regarde d'abord si le fichier est déjà là, puis dans votre Google Drive
s'il est monté, et ne télécharge que s'il ne l'a trouvé nulle part. Si vous avez
monté votre Drive, il y dépose une copie pour la prochaine session.

Premier appel : quelques secondes. Appels suivants : instantané.

In [ ]:
annonces = charger("airbnb_raw")
annonces.head(2)

### Cas 3 : votre OneDrive UCLouvain

Vous disposez tous d'un OneDrive UCLouvain. C'est là que vous mettrez des
données qui ne peuvent pas être publiques.

Le principe est le même que le cas 1, il faut juste transformer le lien de
partage en lien de **téléchargement direct**, sans quoi vous recevrez une page
web au lieu d'un fichier.

1. Déposez le fichier sur votre OneDrive.
2. `Partager` > `Copier le lien`. Vérifiez qui a le droit d'ouvrir le lien.
3. Ajoutez `&download=1` à la fin du lien (ou `?download=1` s'il n'y a pas
   déjà de `?`).
4. `pd.read_csv(lien_modifie)`.

```python
lien = "https://uclouvain-my.sharepoint.com/:x:/g/personal/.../XYZ?e=AbCdEf"
lien_direct = lien.split("?")[0] + "?download=1"
donnees = pd.read_csv(lien_direct)
```

**Attention.** Si le fichier contient des données confidentielles, un lien
ouvert à « toute personne disposant du lien » est un lien public. Ce n'est pas
parce qu'il est long et illisible qu'il est protégé. Pour de vraies données
sensibles, on passe par un partage nominatif et un téléchargement manuel.

### Hack-Time 🛠️

C'est le hack-time le plus utile de la journée. Vous allez faire circuler un
fichier par votre propre OneDrive, du début à la fin.

1. Dans la cellule ci-dessous, **exportez** un petit extrait des annonces vers
   un fichier CSV local. Une piste :
   `annonces.head(50).to_csv("extrait.csv", index=False)`.
2. Téléchargez ce fichier depuis Colab : panneau `Fichiers` à gauche, clic droit
   sur `extrait.csv`, `Télécharger`.
3. Déposez-le sur votre **OneDrive UCLouvain**, créez un lien de partage, et
   convertissez-le en lien de téléchargement direct.
4. Rechargez-le avec `pd.read_csv` et vérifiez que vous retrouvez bien 50 lignes.

**Huit minutes.** Prévenez si l'étape 4 échoue : selon les réglages de
l'UCLouvain, le partage anonyme peut être bloqué, et nous passerons alors par le
Google Drive.

**Défi optionnel**
Que se passe-t-il si vous oubliez le `?download=1` ? Regardez ce que
`pd.read_csv` vous renvoie comme erreur, et expliquez ce que le serveur a
réellement envoyé.

In [ ]:
# Votre code ici

## À retenir

- Un projet rangé : `README.md`, `requirements.txt`, `.gitignore`, `data/`,
  `notebooks/`, `src/`. Numérotez ce qui a un ordre.
- Avant tout rendu : `Redémarrer et tout exécuter`. Si ça casse, ce n'est pas
  prêt. C'est un critère de correction.
- **Aucune donnée dans un dépôt git.** Poids, diffs illisibles, licences, RGPD.
  Le `.gitignore` applique la règle à votre place.
- Trois sources selon le cas : une adresse web pour du public, votre Google
  Drive pour votre travail, votre OneDrive UCLouvain pour du confidentiel.
- Un notebook ouvert depuis GitHub dans Colab est éphémère. Tout ce que vous
  voulez conserver doit être écrit ailleurs que dans la session.

**La suite** : le notebook 2 passe à Git et GitHub, et vous fera pousser votre
premier commit. C'est ce dont vous aurez besoin pour rendre le devoir 1.